# VAE model training
#### Payload type: GPU training + DataLoader

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.ndimage import rotate
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import mediapy as media


In [ ]:
# Load the MNIST dataset
train_dataset = MNIST(root='./data', train=True, download=True, transform=ToTensor())
test_dataset = MNIST(root='./data', train=False, download=True, transform=ToTensor())

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Define the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
## Solution
# Define the VAE model.
class VAE(nn.Module):
	def __init__(self, latent_dim, input_shape=(1, 28, 28)):
		super(VAE, self).__init__()
		flatten_dim = 8 * input_shape[1]//4 * input_shape[2]//4

		self.encoder = nn.Sequential(
            nn.Conv2d(input_shape[0], 16, kernel_size=3, stride=2, padding=1),#1x28x28 -> 16x14x14
            nn.ReLU(),
            nn.Conv2d(16, 8, kernel_size=3, stride=2, padding=1),#16x14x14 -> 8x7x7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(flatten_dim, 64),#8x7x7 -> 64
            nn.ReLU(),
            nn.Linear(64, 2*latent_dim)#64 -> 2*latent_dim
        )

		self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),#latent_dim -> 64
            nn.ReLU(),
            nn.Linear(64, flatten_dim),#64 -> 8x7x7
            nn.ReLU(),
            nn.Unflatten(1, (8, input_shape[1] // 4, input_shape[2] // 4)), #8x7x7 -> 8x7x7
            nn.ConvTranspose2d(8, 16, kernel_size=3, stride=2, padding=1, output_padding=1),#8x7x7 -> 16x14x14
            nn.ReLU(),
            nn.ConvTranspose2d(16, input_shape[0], kernel_size=3, stride=2, padding=1, output_padding=1),#16x14x14 -> 1x28x28
            nn.Sigmoid()
        )

	def encode(self, x):
		h = self.encoder(x)
		mu, log_var = torch.chunk(h, 2, dim=1)  # Split into mean and log variance
		return mu, log_var

	def reparameterize(self, mu, log_var):
		std = torch.exp(0.5 * log_var)
		eps = torch.randn_like(std)
		z = mu + eps * std
		return z

	def decode(self, z):
		return self.decoder(z)

	def forward(self, x):
		mu, log_var = self.encode(x)
		z = self.reparameterize(mu, log_var)
		x_hat = self.decode(z)
		return x_hat, mu, log_var

# Define the loss function.
def vae_loss(x, x_hat, mu, log_var):
	reconstruction_loss = nn.BCELoss(reduction='sum')(x_hat, x)
	kl_divergence = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
	return reconstruction_loss + kl_divergence



In [ ]:
# Initialize VAE model
latent_dim = 2
vae = VAE(latent_dim).to(device)

# Define optimizer
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
	total_loss = 0
	for batch_idx, (data, _) in enumerate(train_loader):
		data = data.to(device)

		optimizer.zero_grad()
		recon_batch, mu, log_var = vae(data)
		loss = vae_loss(data, recon_batch, mu, log_var)
		loss.backward()
		total_loss += loss.item()
		optimizer.step()

	print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {total_loss/len(train_loader.dataset):.4f}")